In [4]:
from google import genai
from google.genai import types
import wave
from loguru import logger
from pathlib import Path
import json

# load the data 

In [6]:
BASE_DIR = Path("__file__").resolve().parent

In [7]:
BASE_DIR

WindowsPath('D:/GAN_AI/Synthetic-Speech-Data-Pipeline-For-STT/notebooks')

In [39]:
DATA_PATH = Path(r"..\data\synthetic_audio_dataset.jsonl")

NOISE_PATH = Path(r"..\data\background_noise")
DATA_PATH

WindowsPath('../data/synthetic_audio_dataset.jsonl')

In [40]:
NOISE_PATH/"sd.wav"

WindowsPath('../data/background_noise/sd.wav')

In [41]:

audio_data = []

with open(DATA_PATH, "r" , encoding="utf-8") as f:
    for line in f:
        audio_data.append(json.loads(line))

print(audio_data)

[{'audio_id': '6d383aa7-5c62-400c-8113-caba39a71aeb', 'prompt_id': '6d383aa7-5c62-400c-8113-caba39a71aeb', 'audio_path': 'data\\audio_outputs\\6d383aa7-5c62-400c-8113-caba39a71aeb.wav', 'text': 'ايه الاخبار يا باشا؟ عامل ايه النهاردة؟', 'voice_name': 'Algenib', 'speaker_id': '3', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'clean audio', 'sample_rate': 24000, 'status': 'generated', 'review_status': 'pending'}, {'audio_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'prompt_id': '759c7a5b-78a7-4749-aeb1-6122cb88b9ec', 'audio_path': 'data\\audio_outputs\\759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav', 'text': 'لو سمحت، هو البنطلون ده بكام؟', 'voice_name': 'Algenib', 'speaker_id': '3', 'tts_model': 'gemini-2.5-flash-preview-tts', 'background_noise': 'background street noise', 'sample_rate': 24000, 'status': 'generated', 'review_status': 'pending'}, {'audio_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'prompt_id': 'ef2b9043-9163-4aa8-8289-8ff1e22ce3f5', 'audio_path': 'data\\au

In [42]:
Path(audio_data[1]['audio_path'])

WindowsPath('data/audio_outputs/759c7a5b-78a7-4749-aeb1-6122cb88b9ec.wav')

In [43]:
import numpy as np
import soundfile as sf
import random
from scipy.signal import resample_poly


def add_background_noise(
    speech_path: str,
    noise_path: str,
    output_path: str,
    snr_db: float = 10,
):
    # Load
    speech, sr = sf.read(speech_path)
    noise, noise_sr = sf.read(noise_path)

    # Mono
    if speech.ndim > 1:
        speech = speech.mean(axis=1)

    if noise.ndim > 1:
        noise = noise.mean(axis=1)

    speech = speech.astype(np.float32)
    noise = noise.astype(np.float32)

    # 🚀 FAST resampling (key speed improvement)
    if noise_sr != sr:
        gcd = np.gcd(noise_sr, sr)
        noise = resample_poly(noise, sr // gcd, noise_sr // gcd)

    # Repeat if needed
    if len(noise) < len(speech):
        repeats = (len(speech) // len(noise)) + 1
        noise = np.tile(noise, repeats)

    # Random crop
    start = random.randint(0, len(noise) - len(speech))
    noise = noise[start:start + len(speech)]

    # Power (can be slightly optimized later, but fine)
    speech_power = np.mean(speech * speech)
    noise_power = np.mean(noise * noise)

    scale = np.sqrt(
        speech_power / (10 ** (snr_db / 10) * noise_power + 1e-9)
    )

    noise *= scale

    mixed = speech + noise

    # Normalize safely
    peak = np.max(np.abs(mixed)) + 1e-9
    mixed = mixed / peak

    sf.write(output_path, mixed, sr)

    return output_path

In [44]:
NOISE_PATH / "crowd_noise.wav"

WindowsPath('../data/background_noise/crowd_noise.wav')

In [50]:
audio_data[1]['background_noise']

'background street noise'

In [61]:
if audio_data[1]['background_noise'] == "background street noise":
    noise_file = "street_noise.wav"
elif audio_data[1]['background_noise'] == "background crowd":
    noise_file = "crowd_noise.wav" 
else:
    pass



add_background_noise(
    speech_path=".." / Path( audio_data[1]['audio_path']),
    noise_path= NOISE_PATH / noise_file ,
    output_path=audio_data[1]['audio_path'].split("\\")[-1].split(".")[0] + "_with_noise.wav",
    snr_db=10,
)

'759c7a5b-78a7-4749-aeb1-6122cb88b9ec_with_noise.wav'